# I2S VAD Research — Model Evaluation

This notebook:
1. Loads all three saved models from `outputs/models/`
2. Runs each model on the test set
3. Computes accuracy, precision, recall, F1, confusion matrix
4. Measures inference latency
5. Saves all figures as PDF to `outputs/figures/`
6. Saves final metrics CSV to `outputs/stats/final_metrics.csv`

**Run all cells top to bottom.**

In [ ]:
import sys
import time
import json
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 150
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# add project root to path
ROOT = Path('').resolve()
if 'notebooks' in str(ROOT):
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.models.cnn1d         import CNN1D
from src.models.wavenet_small import WaveNetSmall
from src.models.ecapa_vad     import ECAPAVAD

# paths
SPLITS     = ROOT / 'data'    / 'splits'
MODELS_DIR = ROOT / 'outputs' / 'models'
STATS_DIR  = ROOT / 'outputs' / 'stats'
FIGS_DIR   = ROOT / 'outputs' / 'figures'
FIGS_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
print(f'ROOT   : {ROOT}')

In [ ]:
# ── Load test set ──────────────────────────────────────────────────────────
X_test = np.load(str(SPLITS / 'X_test.npy'))
y_test = np.load(str(SPLITS / 'y_test.npy'))

X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
y_test_t = torch.tensor(y_test, dtype=torch.long).to(device)

print(f'Test set : X {X_test.shape}  y {y_test.shape}')
print(f'Speech   : {y_test.sum()}  windows')
print(f'Noise    : {(y_test==0).sum()} windows')

In [ ]:
# ── Model registry ────────────────────────────────────────────────────────
MODEL_REGISTRY = {
    'CNN1D':       (CNN1D(num_classes=2),       'cnn1d_best.pt'),
    'WaveNetSmall':(WaveNetSmall(num_classes=2), 'wavenet_best.pt'),
    'ECAPAVAD':    (ECAPAVAD(num_classes=2),     'ecapa_best.pt'),
}

def load_model(model, filename):
    path = MODELS_DIR / filename
    ckpt = torch.load(str(path), map_location=device)
    model.load_state_dict(ckpt['model_state'])
    model = model.to(device)
    model.eval()
    print(f'  Loaded {filename}  '
          f'(best epoch {ckpt["epoch"]}, '
          f'val_acc {ckpt["val_acc"]:.4f})')
    return model

print('Loading models...')
loaded_models = {}
for name, (model, filename) in MODEL_REGISTRY.items():
    loaded_models[name] = load_model(model, filename)
print('All models loaded.')

In [ ]:
# ── Evaluate all models ───────────────────────────────────────────────────
LABELS     = ['Noise', 'Speech']
N_LATENCY  = 200   # number of runs for latency measurement

results = []

for name, model in loaded_models.items():
    print(f'\nEvaluating {name}...')

    # ── predictions ───────────────────────────────────────────────────
    with torch.no_grad():
        logits = model(X_test_t)
        preds  = logits.argmax(dim=1).cpu().numpy()

    y_true = y_test

    # ── metrics ───────────────────────────────────────────────────────
    acc  = accuracy_score(y_true, preds)
    prec = precision_score(y_true, preds, average='macro', zero_division=0)
    rec  = recall_score(y_true, preds, average='macro', zero_division=0)
    f1   = f1_score(y_true, preds, average='macro', zero_division=0)

    # per-class f1
    f1_per = f1_score(y_true, preds, average=None, zero_division=0)

    # ── latency (single window inference) ─────────────────────────────
    single = X_test_t[0:1]   # one window
    # warmup
    for _ in range(10):
        with torch.no_grad():
            _ = model(single)
    # measure
    t0 = time.perf_counter()
    for _ in range(N_LATENCY):
        with torch.no_grad():
            _ = model(single)
    latency_ms = (time.perf_counter() - t0) / N_LATENCY * 1000

    # ── model size ────────────────────────────────────────────────────
    pt_path   = MODELS_DIR / MODEL_REGISTRY[name][1]
    size_kb   = pt_path.stat().st_size / 1024
    n_params  = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f'  Accuracy  : {acc:.4f}')
    print(f'  Precision : {prec:.4f}')
    print(f'  Recall    : {rec:.4f}')
    print(f'  F1 macro  : {f1:.4f}')
    print(f'  F1 noise  : {f1_per[0]:.4f}  |  F1 speech: {f1_per[1]:.4f}')
    print(f'  Latency   : {latency_ms:.3f} ms/window')
    print(f'  Model size: {size_kb:.1f} KB')
    print(f'  Params    : {n_params:,}')

    results.append({
        'model':        name,
        'accuracy':     round(acc,  4),
        'precision':    round(prec, 4),
        'recall':       round(rec,  4),
        'f1_macro':     round(f1,   4),
        'f1_noise':     round(float(f1_per[0]), 4),
        'f1_speech':    round(float(f1_per[1]), 4),
        'latency_ms':   round(latency_ms, 3),
        'size_kb':      round(size_kb, 1),
        'n_params':     n_params,
    })

df_results = pd.DataFrame(results)
print('\n', df_results.to_string(index=False))

In [ ]:
# ── Save final metrics CSV ────────────────────────────────────────────────
csv_path = STATS_DIR / 'final_metrics.csv'
df_results.to_csv(str(csv_path), index=False)
print(f'Final metrics saved to: {csv_path}')

In [ ]:
# ── Figure 1: Confusion Matrices ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Confusion Matrices — All Models', fontsize=14, fontweight='bold')

for ax, (name, model) in zip(axes, loaded_models.items()):
    with torch.no_grad():
        preds = model(X_test_t).argmax(dim=1).cpu().numpy()

    cm = confusion_matrix(y_test, preds)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=LABELS, yticklabels=LABELS,
        ax=ax, cbar=False
    )
    row = df_results[df_results['model'] == name].iloc[0]
    ax.set_title(f'{name}\nAcc={row["accuracy"]:.4f}  F1={row["f1_macro"]:.4f}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

plt.tight_layout()
path = FIGS_DIR / 'confusion_matrices.pdf'
plt.savefig(str(path), bbox_inches='tight')
print(f'Saved: {path}')
plt.show()

In [ ]:
# ── Figure 2: Training curves (loss) ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Training & Validation Loss', fontsize=14, fontweight='bold')

name_to_file = {
    'CNN1D':        'cnn1d_history.json',
    'WaveNetSmall': 'wavenet_history.json',
    'ECAPAVAD':     'ecapa_history.json',
}

for ax, (name, fname) in zip(axes, name_to_file.items()):
    with open(str(STATS_DIR / fname)) as f:
        h = json.load(f)
    epochs = range(1, len(h['train_loss']) + 1)
    ax.plot(epochs, h['train_loss'], label='Train loss', color='steelblue')
    ax.plot(epochs, h['val_loss'],   label='Val loss',   color='tomato')
    ax.set_title(name)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
path = FIGS_DIR / 'training_curves_loss.pdf'
plt.savefig(str(path), bbox_inches='tight')
print(f'Saved: {path}')
plt.show()

In [ ]:
# ── Figure 3: Training curves (accuracy) ─────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Training & Validation Accuracy', fontsize=14, fontweight='bold')

for ax, (name, fname) in zip(axes, name_to_file.items()):
    with open(str(STATS_DIR / fname)) as f:
        h = json.load(f)
    epochs = range(1, len(h['train_acc']) + 1)
    ax.plot(epochs, h['train_acc'], label='Train acc', color='steelblue')
    ax.plot(epochs, h['val_acc'],   label='Val acc',   color='tomato')
    ax.set_title(name)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
path = FIGS_DIR / 'training_curves_acc.pdf'
plt.savefig(str(path), bbox_inches='tight')
print(f'Saved: {path}')
plt.show()

In [ ]:
# ── Figure 4: Model comparison bar chart ─────────────────────────────────
metrics   = ['accuracy', 'precision', 'recall', 'f1_macro']
bar_labels= ['Accuracy', 'Precision', 'Recall', 'F1 Macro']
colors    = ['steelblue', 'seagreen', 'tomato']
x         = np.arange(len(metrics))
width     = 0.25

fig, ax = plt.subplots(figsize=(10, 5))

for i, (_, row) in enumerate(df_results.iterrows()):
    vals = [row[m] for m in metrics]
    bars = ax.bar(x + i * width, vals, width,
                  label=row['model'], color=colors[i], alpha=0.85)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7)

ax.set_xticks(x + width)
ax.set_xticklabels(bar_labels)
ax.set_ylim(0, 1.08)
ax.set_ylabel('Score')
ax.set_title('Model Comparison — Test Set Metrics', fontweight='bold')
ax.legend()
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
path = FIGS_DIR / 'model_comparison_bar.pdf'
plt.savefig(str(path), bbox_inches='tight')
print(f'Saved: {path}')
plt.show()

In [ ]:
# ── Figure 5: Accuracy vs Latency vs Size tradeoff ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Edge Deployment Tradeoff Analysis', fontsize=13, fontweight='bold')

# left: accuracy vs latency
ax = axes[0]
for i, row in df_results.iterrows():
    ax.scatter(row['latency_ms'], row['accuracy'],
               s=200, color=colors[i], zorder=5, label=row['model'])
    ax.annotate(row['model'],
                (row['latency_ms'], row['accuracy']),
                textcoords='offset points', xytext=(8, 4), fontsize=9)

# real-time threshold line (32ms window = must finish within 32ms)
ax.axvline(x=32, color='red', linestyle='--', alpha=0.6, label='32ms real-time limit')
ax.set_xlabel('Inference Latency (ms)')
ax.set_ylabel('Test Accuracy')
ax.set_title('Accuracy vs Latency')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# right: accuracy vs model size
ax = axes[1]
for i, row in df_results.iterrows():
    ax.scatter(row['size_kb'], row['accuracy'],
               s=200, color=colors[i], zorder=5, label=row['model'])
    ax.annotate(row['model'],
                (row['size_kb'], row['accuracy']),
                textcoords='offset points', xytext=(8, 4), fontsize=9)

ax.set_xlabel('Model Size (KB)')
ax.set_ylabel('Test Accuracy')
ax.set_title('Accuracy vs Model Size')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
path = FIGS_DIR / 'tradeoff_analysis.pdf'
plt.savefig(str(path), bbox_inches='tight')
print(f'Saved: {path}')
plt.show()

In [ ]:
# ── Figure 6: Summary table as figure ────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 3))
ax.axis('off')

col_labels = ['Model', 'Accuracy', 'Precision', 'Recall',
              'F1 Macro', 'F1 Noise', 'F1 Speech',
              'Latency (ms)', 'Size (KB)', 'Params']

table_data = []
for _, row in df_results.iterrows():
    table_data.append([
        row['model'],
        f"{row['accuracy']:.4f}",
        f"{row['precision']:.4f}",
        f"{row['recall']:.4f}",
        f"{row['f1_macro']:.4f}",
        f"{row['f1_noise']:.4f}",
        f"{row['f1_speech']:.4f}",
        f"{row['latency_ms']:.3f}",
        f"{row['size_kb']:.1f}",
        f"{row['n_params']:,}",
    ])

table = ax.table(
    cellText=table_data,
    colLabels=col_labels,
    cellLoc='center',
    loc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.2, 1.8)

# style header
for j in range(len(col_labels)):
    table[0, j].set_facecolor('#2c3e50')
    table[0, j].set_text_props(color='white', fontweight='bold')

# alternate row colors
for i in range(1, len(table_data) + 1):
    color = '#eaf0fb' if i % 2 == 0 else 'white'
    for j in range(len(col_labels)):
        table[i, j].set_facecolor(color)

ax.set_title('Model Comparison Summary Table',
             fontsize=13, fontweight='bold', pad=20)

plt.tight_layout()
path = FIGS_DIR / 'summary_table.pdf'
plt.savefig(str(path), bbox_inches='tight')
print(f'Saved: {path}')
plt.show()

In [ ]:
# ── Final summary ─────────────────────────────────────────────────────────
print('='*55)
print('  EVALUATION COMPLETE')
print('='*55)
print(f'\nFiles saved to outputs/figures/:')
for f in sorted(FIGS_DIR.glob('*.pdf')):
    print(f'  {f.name}')
print(f'\nFinal metrics CSV:')
print(f'  {STATS_DIR / "final_metrics.csv"}')
print(f'\nShare final_metrics.csv for paper writing.')
print()
print(df_results.to_string(index=False))